In [1]:
pip install speechrecognition ollama pyttsx3 pyaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 47.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)


In [4]:
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Install portaudio19-dev for PyAudio in Colab
    !apt-get install -y portaudio19-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libasound2-dev libjack-dev libjack0 libportaudio2 libportaudiocpp0
Suggested packages:
  libasound2-doc jackd1 portaudio19-doc
The following packages will be REMOVED:
  libjack-jackd2-0
The following NEW packages will be installed:
  libasound2-dev libjack-dev libjack0 libportaudio2 libportaudiocpp0
  portaudio19-dev
0 upgraded, 6 newly installed, 1 to remove and 3 not upgraded.
Need to get 596 kB of archives.
After this operation, 3,178 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjack0 amd64 1:0.125.0-3build2 [93.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libasound2-dev amd64 1.2.6.1-1ubuntu1.1 [110 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjack-dev amd64 1:0.125.0-3build2 [206 kB]
Get:4 http://archive.ubuntu.com/ubuntu jamm

In [5]:
pip install speechrecognition ollama pyttsx3 pyaudio

  Using cached speechrecognition-3.16.1-py3-none-any.whl.metadata (28 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pyttsx3-2.99-py3-none-any.whl.metadata (6.2 kB)
  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached speechrecognition-3.16.1-py3-none-any.whl (32.9 MB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
Using cached pyttsx3-2.99-py3-none-any.whl (32 kB)
  Created wheel for pyaudio: filename=pyaudio-0.2.14-cp312-cp312-linux_x86_64.whl size=68676 sha256=747a71fb5f9022fd9b36c7bd117b6d17efdb439e20c4d273f60ea5aee8493fb7
  Stored in directory: /root/.cache/pip/wheels/68/c7/33/c6a6b210cb5819ec5c219928c794a447742a7d86d21c0b92e6
Successfully built pyaudio


# Step 1: Importing the Tools

In [6]:
import speech_recognition as sr
import ollama
import pyttsx3

# Step 2: The Ears

In [7]:
def listen():
    recognizer = sr.Recognizer()

    try:
        with sr.Microphone() as source:
            print("Listening... (Speak now)")
            # Adjust for ambient noise
            recognizer.adjust_for_ambient_noise(source, duration=0.5)

            # Listen for audio input
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)
            print("Processing...")

        # Recognize speech using Google's free API
        text = recognizer.recognize_google(audio)
        print(f"You said: {text}")
        return text

    except sr.WaitTimeoutError:
        print("No speech detected (timeout).")
        return None
    except sr.UnknownValueError:
        print("Sorry, I didn't catch that.")
        return None
    except sr.RequestError:
        print("Speech recognition service unavailable.")
        return None
    except Exception as e:
        print(f"An error occurred in listen(): {e}")
        return None

adjust_for_ambient_noise: Microphones pick up fan hums and static. This line tells the code to listen to the silence for 0.5 seconds to understand the room’s baseline noise, which makes the actual recognition much more accurate.
recognize_google:using Google’s Web Speech API to convert audio to text. It’s free and generally very accurate, though it does require an internet connection.

# Step 3: The Brain

In [8]:
def think(text: str):
    if not text:
        return None

    print("Thinking...")

    try:
        # Ensure you have pulled the model via: ollama pull llama3
        response = ollama.chat(
            model="llama3",
            messages=[
                {
                    "role": "user",
                    "content": text,
                }
            ],
        )

        response_text = response["message"]["content"]
        print(f"AI: {response_text}")
        return response_text

    except Exception as e:
        print(f"An error occurred in think(): {e}")
        return "Sorry, something went wrong while thinking."

ollama.chat: This is the interface to the local Llama 3 model. sending a list of messages (in this case, just one from the “user”) and wait for the model to complete the pattern.
Latency: Since Llama 3 is running locally on the device, this might take a second or two, depending on your GPU/CPU, but it’s completely private. No data is sent to a cloud server for thinking.

# Step 4: The Mouth

In [9]:
def speak(text: str):
    if not text:
        return

    try:
        engine = pyttsx3.init()

        # Optional: Change voice properties
        voices = engine.getProperty("voices")
        if voices:
            # Try changing index 0 -> 1 for alternative voice
            engine.setProperty("voice", voices[0].id)

        engine.setProperty("rate", 175)  # Speed of speech

        engine.say(text)
        engine.runAndWait()

    except Exception as e:
        print(f"An error occurred in speak(): {e}")

pyttsx3.init(): This initialises the speech engine driver on the OS (sapi5 on Windows, nsss on Mac, espeak on Linux).
engine.runAndWait(): This is critical. It blocks the code execution until the speaking is done. Without this, the program might try to listen while it’s still speaking, causing it to hear itself!

# Step 5: The Main Function

In [ ]:
def main():
    print("--- Voice Assistant Started ---")
    speak("Hello, I am ready. You can start speaking.")

    while True:
        # 1. Listen
        user_input = listen()

        # Skip if nothing heard
        if not user_input:
            continue

        # 2. Check for exit keywords
        if user_input.lower().strip() in ["exit", "stop", "quit"]:
            speak("Goodbye!")
            print("Exiting...")
            break

        # 3. Think
        ai_response = think(user_input)

        # 4. Speak
        speak(ai_response)


if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Default Input Device Available
An error occurred in listen(): No Defau

The while True Loop: This creates the always-on behaviour. The program enters a cycle of Listen -> Think -> Speak, and then immediately goes back to Listen.
Exit Strategy:  added a simple check for exit or stop so you can gracefully shut down the assistant without force-quitting the terminal.